# GFQL Semantic Operators: `sem_decide` and `sem_map`

Narrow a graph with GFQL, then ask a model typed questions about the rows that remain. The answers come back as columns next to
your data, with probabilities when the provider can measure them, and a trace tells you what the run did and what it cost.

This notebook runs **offline**: Python functions and a mock HTTP session play the model, so it needs no API key and sends no data
anywhere. The last cells show live provider templates. They stay off until you turn them on.

| you will see | where |
|---|---|
| the whole workflow in one Cypher query, with `$params` | **Quickstart** |
| the same question as a `WHERE` term, in the fluent form, as a GFQL chain and as a saved `call(...)` | **Four ways to call** |
| typed questions and how to read the result safely | **Questions**, **Read the result** |
| probabilities, `min_p`, `sem_map` | **Probabilities**, **Text with `sem_map`** |
| plan before, trace after, `dedupe`, a result cache, guardrails | **Plan, run, trace** |
| Polars tables | **Polars** |
| HTTP providers (OpenAI Decisions, chat, Anthropic, LangChain) and live templates | **HTTP providers**, **Live providers** |

The reference for every parameter is the *Semantic Table Operators* section of the
[GFQL built-in calls](https://pygraphistry.readthedocs.io/en/latest/gfql/builtin_calls.html) page.

## Setup: sample data and a stand-in for a model

The sample graph holds security alerts. One alert has no text, so you can see how the operators treat empty evidence. `rules` is a
keyword rule that plays the model.

In [1]:
import json
import warnings

import pandas as pd

import graphistry
from graphistry import call, e_forward, n
from graphistry.compute.semantic import (
    AnthropicMessagesProvider, CallableDecisionProvider, CallableGenerationProvider, MemorySemanticCache, OpenAIChatProvider,
    OpenAIDecisionsProvider, TokenPrices, bind_providers, explain_sem_decide, semantic_summary, semantic_trace, semantic_verdicts)

pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 60)

alerts = pd.DataFrame({
    "id": ["a1", "a2", "a3", "a4", "a5"],
    "text": ["Login from new country for admin account, 14 failed attempts first",
             "Nightly backup job finished in 42 minutes",
             "DNS query to xk3-update.top returned NXDOMAIN 300 times",
             "User opened shared spreadsheet from finance team",
             None],
    "host": ["web-1", "db-1", "web-1", "laptop-7", "laptop-9"],
})
links = pd.DataFrame({"src": ["a1", "a3", "a4"], "dst": ["a3", "a1", "a2"]})
g = graphistry.nodes(alerts, "id").edges(links, "src", "dst")


def triage(evidence, question):
    text = str(evidence).lower()
    if question.kind == "predicate":
        return "failed" in text or "nxdomain" in text
    if question.kind == "choice":
        return "network" if "dns" in text else "access" if "login" in text else "routine"
    return "high" if "failed" in text else "medium" if "nxdomain" in text else "low"


rules = CallableDecisionProvider(triage, name="rules", fingerprint="triage-rules-v1")
g._nodes

,id,text,host
0,a1,"Login from new country for admin account, 14 failed atte...",web-1
1,a2,Nightly backup job finished in 42 minutes,db-1
2,a3,DNS query to xk3-update.top returned NXDOMAIN 300 times,web-1
3,a4,User opened shared spreadsheet from finance team,laptop-7
4,a5,NaN,laptop-9


## Quickstart: one Cypher query

One query narrows the graph and asks the model about the rows that remain. The question and the limits are `$params`, so the query
text stays plain and safe to save. `bind_providers` names the provider and the cache outside the query. The `rules` function
plays the model, so this runs offline.

In [2]:
# 1. Config: the question and a cost limit travel as $params (the provider is bound by alias below)
params = {"question": "Does this alert describe suspicious activity?", "max_usd": 0.50}

# 2. Run: one query narrows the graph, then asks the model about the surviving rows
with bind_providers({"default": rules}, cache=MemorySemanticCache()), semantic_trace() as runs:
    answered = g.gfql("""
        MATCH (a {host: 'web-1'})-[]->(b)
        WITH b.id AS id, b.text AS text
        CALL graphistry.sem_decide({
            question: $question,
            evidence: ['text'],
            max_usd: $max_usd
        }) YIELD verdict, score, status
        RETURN id, text, verdict, status
        ORDER BY id
    """, params=params)

# 3. Output: the rows, and what the run did
display(answered._nodes)
{k: runs[-1][k] for k in ("total_rows", "provider_calls", "billed_cost_usd")}

,id,text,verdict,status
0,a1,"Login from new country for admin account, 14 failed atte...",True,ok
1,a3,DNS query to xk3-update.top returned NXDOMAIN 300 times,True,ok


{'total_rows': 2, 'provider_calls': 2, 'billed_cost_usd': 0.0}

### The same question as a WHERE term

`WHERE ... AND sem_decide(question, {name: expression})` is short for the `CALL` form. The structural terms run first, so the
model sees only the rows that pass them. One batched call follows.

In [3]:
with bind_providers({"default": rules}):
    flagged = g.gfql("""
        MATCH (a {host: 'web-1'})-[]->(b)
        WHERE b.text IS NOT NULL AND sem_decide($question, {text: b.text})
        RETURN b.id AS id, b.text AS text
        ORDER BY id
    """, params=params)
flagged._nodes

,id,text
0,a1,"Login from new country for admin account, 14 failed atte..."
1,a3,DNS query to xk3-update.top returned NXDOMAIN 300 times


### The fluent form

`g.gfql("MATCH ...").sem_decide(...)` gives the same rows in Python. Use it when you want several questions, or a provider object
instead of an alias. The cells below use this form.

In [4]:
# 1. Input: narrow the graph with Cypher
candidates = g.gfql("""
    MATCH (a {host: 'web-1'})-[]->(b)
    RETURN b.id AS id, b.text AS text
""")

# 2. Config: typed questions (a rule stands in for the model, see `rules` above)
questions = [
    {
        "id": "suspicious",
        "kind": "predicate",
        "text": "Does this alert describe suspicious activity?",
    },
    {
        "id": "severity",
        "kind": "score",
        "text": "How severe is it?",
        "levels": ["low", "medium", "high"],
    },
]

# 3. Run: semantic decisions on the Cypher result, with a result cache and a trace
with bind_providers({"triage": rules}, cache=MemorySemanticCache()), semantic_trace() as runs:
    triaged = candidates.sem_decide(
        questions,
        "triage",
        task_id="triage",
        evidence_cols=["text"],
    )

# 4. Output: status counts that never hide an unanswered row, and what the run did
display(semantic_summary(triaged._nodes))
display(triaged._nodes[["id", "suspicious", "severity", "severity__status"]])
{k: runs[-1][k] for k in ("total_rows", "provider_calls", "billed_cost_usd", "wall_ms")}

,ok,uncertain,error,skipped,unanswered,unanswered_pct
question,,,,,,
suspicious,2,0,0,0,0,0.0
severity,2,0,0,0,0,0.0


,id,suspicious,severity,severity__status
0,a1,True,high,ok
2,a3,True,medium,ok


{'total_rows': 2,
 'provider_calls': 2,
 'billed_cost_usd': 0.0,
 'wall_ms': 4.176815040409565}

## Four ways to call

The forms answer the same question. The `WHERE` form filters, and the `CALL` form keeps every row and its status. Pick the one that fits your code.

| form | when to use it |
|---|---|
| `g.gfql("MATCH ... WITH ... CALL graphistry.sem_decide({...}) YIELD ... RETURN ...", params=...)` | one query from input to output, with `$params` and a bound provider alias (the quickstart above) |
| `g.gfql("MATCH ... WHERE ... AND sem_decide(question, {...})")` | a single yes/no filter after cheap structural terms |
| `g.gfql("MATCH ... RETURN ...").sem_decide(...)` | Python code that holds several questions or a provider object. Alias the returned columns to plain names |
| `g.gfql([n(...), ..., call("sem_decide", {...})])` | a chain of GFQL objects, or a query that you save as JSON: it names the provider by alias and holds no key |

In [5]:
# The same query as a GFQL chain
chain = g.gfql([n({"host": "web-1"}), e_forward(), n()])
via_chain = chain.sem_decide(questions[:1], rules, task_id="triage", evidence_cols=["text"])
via_chain._nodes[["id", "suspicious", "suspicious__status"]]

,id,suspicious,suspicious__status
0,a1,True,ok
2,a3,True,ok


In [6]:
# The same query as a saved GFQL call: the provider is named by an alias, so no key is in the query
query = [
    call(
        "sem_decide",
        {
            "questions": questions[:1],
            "provider": "triage",
            "task_id": "triage",
            "evidence_cols": ["text"],
        },
    )
]
print(json.dumps([step.to_json() for step in query])[:110], "...")

with bind_providers({"triage": rules}):
    via_call = g.gfql([n({"host": "web-1"}), e_forward(), n()] + query)
via_call._nodes[["id", "suspicious", "suspicious__status"]]

[{"type": "Call", "function": "sem_decide", "params": {"questions": [{"id": "suspicious", "kind": "predicate", ...


,id,suspicious,suspicious__status
0,a1,True,ok
2,a3,True,ok


## Questions: three kinds

A question has an `id`, a `kind` and a `text`. The `id` names the output columns, so each `id` must be unique.

| kind | answer | extra fields |
|---|---|---|
| `predicate` | yes or no (`True` or `False`) | none |
| `choice` | one of the `options` | `options` |
| `score` | one of the `levels`, from lowest to highest | `levels` |

In [7]:
QUESTIONS = [
    {"id": "suspicious", "kind": "predicate", "text": "Does this alert describe suspicious activity?"},
    {"id": "category", "kind": "choice", "text": "Which category fits best?", "options": ["access", "network", "routine"]},
    {"id": "severity", "kind": "score", "text": "How severe is it?", "levels": ["low", "medium", "high"]},
]
triaged = g.sem_decide(QUESTIONS, rules, task_id="triage", evidence_cols=["text"])
triaged._nodes[["id", "suspicious", "category", "severity", "severity__status"]]

,id,suspicious,category,severity,severity__status
0,a1,True,access,high,ok
1,a2,False,routine,low,ok
2,a3,True,network,medium,ok
3,a4,False,routine,low,ok
4,a5,<NA>,NaN,NaN,skipped


Alert `a5` has no text. The operator does not call the provider for it. The row stays in the table with status `skipped`. A skipped row
is never dropped.

## Read the result

For each question `q`, the operator adds these columns:

| column | meaning |
|---|---|
| `q` | the answer (a nullable boolean for a predicate, text for the other kinds) |
| `q__status` | `ok`, `uncertain`, `error` or `skipped` |
| `q__p` | the probability that the answer distribution gives the answer (missing when there is none) |
| `q__dist` | the whole distribution as JSON, or null |
| `q__probability_kind` | `native_decision`, `token_logprob`, `verbalized` or `none` |
| `q__error_code` | the reason for an `error` row |
| `q__meta` | JSON with model, provider, input hash, cost, latency, retry count and status reason |

An answer that cannot be read is an `error`. The operator never invents a default answer or a distribution.

In [8]:
meta = json.loads(triaged._nodes["suspicious__meta"].iloc[0])
print({k: meta[k] for k in ("provider", "model", "cost_usd", "retry_count", "task_version")})
skipped = triaged._nodes[triaged._nodes["suspicious__status"] == "skipped"]
print("status reason of the skipped row:", json.loads(skipped["suspicious__meta"].iloc[0])["status_reason"])

{'provider': 'rules', 'model': 'rules', 'cost_usd': 0.0, 'retry_count': 0, 'task_version': '1'}
status reason of the skipped row: null_evidence


### Read answers safely

A skipped or failed row has no answer, but an expression such as `severity != "high"` is still true for it. A report that counts that
row as "not high" shows a row as safe when the model never answered it. `semantic_verdicts` keeps `skipped`, `error` and `uncertain`
apart from `yes` and `no`, and `semantic_summary` counts every status.

In [9]:
naive = triaged._nodes["severity"] != "high"                                     # the trap: the skipped row counts as "not high"
verdict = semantic_verdicts(triaged._nodes, "severity", positive="high")        # yes | no | uncertain | error | skipped
print("rows 'not high' by the naive test:", int(naive.sum()), "| rows the model answered 'no':", int((verdict == "no").sum()))
pd.DataFrame({"id": triaged._nodes["id"], "severity": triaged._nodes["severity"], "verdict": verdict})

rows 'not high' by the naive test: 4 | rows the model answered 'no': 3


,id,severity,verdict
0,a1,high,yes
1,a2,low,no
2,a3,medium,no
3,a4,low,no
4,a5,NaN,skipped


## Probabilities and `min_p`

A provider can return a distribution with the answer. Then `q__dist` and `q__p` are set. `min_p` turns an `ok` answer into
`uncertain` when its probability is below the limit, or when it has none.

In [10]:
P_TRUE_BY_ALERT = {"a1": 0.97, "a2": 0.03, "a3": 0.90, "a4": 0.45}
P_TRUE_BY_TEXT = {text: P_TRUE_BY_ALERT[alert_id] for alert_id, text in zip(alerts["id"], alerts["text"]) if isinstance(text, str)}


def with_probability(evidence, question):
    if question.kind != "predicate":
        return triage(evidence, question)
    p_true = P_TRUE_BY_TEXT[evidence["text"]]
    return p_true > 0.5, {"true": p_true, "false": 1 - p_true}


scorer = CallableDecisionProvider(with_probability, name="scorer", probability_kind="native_decision", fingerprint="scorer-v1")
suspicious_only = [QUESTIONS[0]]
plain = g.sem_decide(suspicious_only, scorer, task_id="triage", evidence_cols=["text"])
strict = g.sem_decide(suspicious_only, scorer, task_id="triage", evidence_cols=["text"], min_p=0.8)
pd.DataFrame({
    "id": plain._nodes["id"],
    "answer": plain._nodes["suspicious"],
    "p": plain._nodes["suspicious__p"].round(2),
    "status": plain._nodes["suspicious__status"],
    "status with min_p=0.8": strict._nodes["suspicious__status"],
})

,id,answer,p,status,status with min_p=0.8
0,a1,True,0.97,ok,ok
1,a2,False,0.97,ok,ok
2,a3,True,0.90,ok,ok
3,a4,False,0.55,ok,uncertain
4,a5,<NA>,NaN,skipped,skipped


Alert `a4` is `False` with probability 0.55. With `min_p=0.8` the answer becomes `uncertain`.

`require_probabilities=True` accepts an answer only when its probability is measured: `native_decision` or `token_logprob`.
`'native'` accepts only `native_decision`. A provider that cannot report the kind gives an `error` with code
`probabilities_unavailable`, and the answer is cleared.

In [11]:
measured = g.sem_decide(suspicious_only, scorer, task_id="triage", evidence_cols=["text"], require_probabilities=True)
unmeasured = g.sem_decide(suspicious_only, rules, task_id="triage", evidence_cols=["text"], require_probabilities=True)
pd.DataFrame({
    "id": measured._nodes["id"],
    "measured provider": measured._nodes["suspicious__status"],
    "value-only provider": unmeasured._nodes["suspicious__status"],
    "reason": unmeasured._nodes["suspicious__error_code"],
})

,id,measured provider,value-only provider,reason
0,a1,ok,error,probabilities_unavailable
1,a2,ok,error,probabilities_unavailable
2,a3,ok,error,probabilities_unavailable
3,a4,ok,error,probabilities_unavailable
4,a5,skipped,skipped,NaN


## Text with `sem_map`

`sem_map` takes an `instruction` and an `out_col`. It adds `out_col`, `{out_col}__status`, `{out_col}__error_code` and `{out_col}__meta`.

In [12]:
def summarize(evidence, instruction):
    return str(evidence["text"]).split(",")[0].strip()[:40]


writer = CallableGenerationProvider(summarize, name="summarizer")
summary = g.sem_map("Write a short title for this alert.", "title", writer, task_id="titles", evidence_cols=["text"])
summary._nodes[["id", "title", "title__status"]]

,id,title,title__status
0,a1,Login from new country for admin account,ok
1,a2,Nightly backup job finished in 42 minute,ok
2,a3,DNS query to xk3-update.top returned NXD,ok
3,a4,User opened shared spreadsheet from fina,ok
4,a5,NaN,skipped


## Plan, run, trace

Three tools tell you what a run will do, what it did, and what it saved.

| tool | answers | sends a request? |
|---|---|---|
| `explain_sem_decide(...)`, `explain_sem_map(...)` | what the run would do: rows, calls, estimated cost | never |
| `with semantic_trace() as runs:` | what each run did: calls, billed cost, status counts, time | only the run's own |
| `semantic_summary(...)`, `semantic_verdicts(...)` | which rows the model answered | never |

The table below repeats the same three alerts, so `dedupe` (on by default) asks about each distinct text once.

In [13]:
repeated = pd.DataFrame({"id": [f"r{i}" for i in range(9)], "text": list(alerts["text"][:3]) * 3})
args = dict(questions=suspicious_only, provider=rules, task_id="triage", evidence_cols=["text"])

plan = explain_sem_decide(repeated, **args)                      # no provider call
print({k: plan[k] for k in ("total_rows", "unique_rows", "dedupe_saved_rows", "planned_calls")})

with semantic_trace() as runs:
    out = g.nodes(repeated, "id").sem_decide(suspicious_only, rules, task_id="triage", evidence_cols=["text"])
print({k: runs[-1][k] for k in ("total_rows", "deduped_rows", "provider_calls")}, "-> the plan and the run agree")
out._nodes[["id", "suspicious", "suspicious__status"]].head(4)

{'total_rows': 9, 'unique_rows': 3, 'dedupe_saved_rows': 6, 'planned_calls': 3}
{'total_rows': 9, 'deduped_rows': 6, 'provider_calls': 3} -> the plan and the run agree


,id,suspicious,suspicious__status
0,r0,True,ok
1,r1,False,ok
2,r2,True,ok
3,r3,True,ok


`dedupe=False` asks for every row, for example when you want several answers from a model that is not deterministic.

A **result cache** answers repeated input between runs. You create it, and `bind_providers(..., cache=cache)` gives it to the runs in
a block. The cache holds each answer before `min_p`, so a later run with another `min_p` still hits. A plan reads the cache without
changing it.

In [14]:
cache = MemorySemanticCache(max_entries=1000, ttl_s=900)
with bind_providers({}, cache=cache):
    for label in ("first run", "second run"):
        before = explain_sem_decide(repeated, **args)
        with semantic_trace() as runs:
            g.nodes(repeated, "id").sem_decide(suspicious_only, rules, task_id="triage", evidence_cols=["text"])
        print(f"{label}: plan says cache hits = {before['cache_hits']}, run made {runs[-1]['provider_calls']} provider calls")
{k: round(v, 2) if isinstance(v, float) else v for k, v in cache.stats().items()}

first run: plan says cache hits = 0, run made 3 provider calls
second run: plan says cache hits = 3, run made 0 provider calls


{'hits': 3,
 'misses': 3,
 'puts': 3,
 'evictions': 0,
 'expirations': 0,
 'rejected': 0,
 'entries': 3,
 'bytes': 1354,
 'hit_rate': 0.5}

### Guardrails

These limits stop a run before it grows too large. A row that the limit stops is `skipped` with a status reason. The row is never
dropped.

| parameter | effect |
|---|---|
| `max_rows` | rows at position `max_rows` and after are skipped |
| `budget={"max_requests": n}` | at most `n` provider calls |
| `budget={"max_cost_usd": x}` | the summed *estimated* cost of the calls stays within `x` |
| `deadline_s` | seconds for the whole run |
| `batch_size`, `max_concurrency` | rows per provider call, and calls at the same time |

In [15]:
limited = g.sem_decide(suspicious_only, rules, task_id="triage", evidence_cols=["text"],
                       batch_size=1, budget={"max_requests": 2}, max_rows=4)
reasons = [json.loads(m)["status_reason"] for m in limited._nodes["suspicious__meta"]]
pd.DataFrame({"id": limited._nodes["id"], "status": limited._nodes["suspicious__status"], "reason": reasons})

,id,status,reason
0,a1,ok,NaN
1,a2,ok,NaN
2,a3,skipped,budget
3,a4,skipped,budget
4,a5,skipped,max_rows


## Polars tables

A Polars graph runs `sem_decide` through GFQL with the off-engine bridge. The call runs on pandas and the result returns to Polars
through Arrow. GFQL shows this as a `RuntimeWarning`, once for each function in a session. A direct `g.sem_decide(...)` call on a Polars
graph returns pandas tables.

To refuse the bridge, call `set_call_mode('strict')` or set the `GFQL_POLARS_CALL_MODE=strict` environment variable. Then
`engine='polars'` raises `NotImplementedError`, and `engine='auto'` runs the query on pandas.

In [16]:
try:
    import polars as pl
except ImportError:
    pl = None
    print("polars is not installed; skip the Polars cells")

if pl is not None:
    g_pl = graphistry.nodes(pl.from_pandas(alerts), "id").edges(pl.from_pandas(links), "src", "dst")
    with bind_providers({"triage": rules}), warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always")
        via_bridge = g_pl.gfql(query)
    print("through GFQL:", type(via_bridge._nodes).__module__)
    print("warning     :", [str(w.message)[:95] for w in caught if "off-engine" in str(w.message)] or "shown earlier in this session")
    direct = g_pl.sem_decide(suspicious_only, rules, task_id="triage", evidence_cols=["text"])
    print("direct call :", type(direct._nodes).__module__)

through GFQL: polars.dataframe.frame
direct call : pandas


In [17]:
from graphistry.compute.gfql.lazy import set_call_mode

if pl is not None:
    set_call_mode("strict")
    try:
        with bind_providers({"triage": rules}):
            try:
                g_pl.gfql(query, engine="polars")
            except NotImplementedError as error:
                print("engine='polars':", type(error).__name__, "-", str(error)[:70])
            on_pandas = g_pl.gfql(query, engine="auto")
            print("engine='auto'  :", type(on_pandas._nodes).__module__)
    finally:
        set_call_mode(None)

engine='polars': NotImplementedError - GFQL engine='polars' does not natively support call 'sem_decide' (it r
engine='auto'  : pandas


## HTTP providers with a mock session

The HTTP providers take a `session` object with a `requests`-style `post` method. A mock session shows what a provider sends and how it
reads a reply, with no network. The next cell holds the mock plumbing (a replaying session and reply builders for the OpenAI
Decisions API, OpenAI chat and Anthropic). **Skim or skip it.** The cells after it use the mock.

In [18]:
import math


class MockResponse:
    def __init__(self, status, body, headers=None):
        self.status_code, self._body, self.headers = status, body, headers or {}

    def json(self):
        return self._body


class MockSession:
    """Replays replies in order (the last one repeats) and records each request body."""

    def __init__(self, replies):
        self.replies, self.posts = list(replies), []

    def post(self, url, *, json, headers, timeout):
        self.posts.append(json)
        reply = self.replies.pop(0) if len(self.replies) > 1 else self.replies[0]
        return reply(json) if callable(reply) else reply


def decisions_reply(body):                     # OpenAI Decisions API: native probabilities
    answers = []
    for q in body["questions"]:
        record = (body["input"] + " " + q["instructions"]).lower()
        p_true = 0.92 if "failed" in record or "nxdomain" in record else 0.08
        answers.append({"type": "predicate", "name": q["name"], "probability": p_true})
    usage = {"input_tokens": 400, "input_tokens_details": {"cached_tokens": 0}, "output_tokens": 20}
    return MockResponse(200, {"model": "mock-decisions", "answers": answers, "usage": usage})


def logprob_reply(letter, alternatives):       # OpenAI chat with token logprobs
    top = [{"token": token, "logprob": math.log(p)} for token, p in alternatives.items()]
    choice = {"message": {"role": "assistant", "content": letter},
              "logprobs": {"content": [{"token": letter, "logprob": math.log(alternatives[letter]), "top_logprobs": top}]}}
    usage = {"prompt_tokens": 120, "completion_tokens": 1, "prompt_tokens_details": {"cached_tokens": 0}}
    return MockResponse(200, {"model": "mock-chat", "choices": [choice], "usage": usage})


def text_reply(text):                          # OpenAI chat, free text
    usage = {"prompt_tokens": 100, "completion_tokens": 3, "prompt_tokens_details": {"cached_tokens": 0}}
    return MockResponse(200, {"model": "mock-chat", "choices": [
        {"message": {"role": "assistant", "content": text}, "finish_reason": "stop"}], "usage": usage})


def anthropic_reply(text):                     # Anthropic Messages
    return MockResponse(200, {"model": "mock-claude", "stop_reason": "end_turn", "content": [{"type": "text", "text": text}],
                              "usage": {"input_tokens": 90, "output_tokens": 12}})


prices = TokenPrices(input=2.0, cached_input=0.5, per_question=0.001)       # your prices per million tokens; there are no defaults

In [19]:
session = MockSession([decisions_reply])
openai_decisions = OpenAIDecisionsProvider("not-a-real-key", "mock-model", prices=prices, session=session)
rows = g.sem_decide(suspicious_only, openai_decisions, task_id="triage", evidence_cols=["text"])
print("HTTP requests:", len(session.posts), "| kind:", rows._nodes["suspicious__probability_kind"].iloc[0])
rows._nodes[["id", "suspicious", "suspicious__p", "suspicious__status"]]

HTTP requests: 4 | kind: native_decision


,id,suspicious,suspicious__p,suspicious__status
0,a1,True,0.92,ok
1,a2,False,0.92,ok
2,a3,True,0.92,ok
3,a4,False,0.92,ok
4,a5,<NA>,NaN,skipped


### Cost and cost cap

Each row carries its share of the billed cost in `q__meta`. `budget={"max_cost_usd": x}` caps the *estimated* cost of the calls before
they are sent. The billed cost can differ from the estimate. The trace sums the billed cost for you.

In [20]:
with semantic_trace() as runs:
    session = MockSession([decisions_reply])
    capped = g.sem_decide(suspicious_only, OpenAIDecisionsProvider("not-a-real-key", "mock-model", prices=prices, session=session),
                          task_id="triage", evidence_cols=["text"], batch_size=1, budget={"max_cost_usd": 0.0025})
stats = runs[-1]
print(f"HTTP requests: {len(session.posts)} | billed ${stats['billed_cost_usd']:.6f} | rows stopped by the cap: {stats['budget_skipped_rows']}")
capped._nodes[["id", "suspicious__status"]]

HTTP requests: 2 | billed $0.001600 | rows stopped by the cap: 2


,id,suspicious__status
0,a1,ok
1,a2,ok
2,a3,skipped
3,a4,skipped
4,a5,skipped


### Pack rows into one request

`OpenAIDecisionsProvider` sends one request per row by default. `rows_per_request=k` packs up to `k` rows into one request. Each question
carries its own row's evidence. Packing cuts the number of requests and the latency. It does not cut tokens. A packed answer records
`extra.packed_rows`. One planned call is one HTTP request, so `budget.max_requests` counts the requests that are sent.

In [21]:
session = MockSession([decisions_reply])
packed = OpenAIDecisionsProvider("not-a-real-key", "mock-model", prices=prices, session=session, rows_per_request=4)
out = g.sem_decide(suspicious_only, packed, task_id="triage", evidence_cols=["text"])
print("HTTP requests:", len(session.posts), "| rows in the first request:", len(session.posts[0]["questions"]))
print("packed_rows:", json.loads(out._nodes["suspicious__meta"].iloc[0])["extra"]["packed_rows"])

HTTP requests: 1 | rows in the first request: 4
packed_rows: 4


## Chat providers: logprobs, structured replies and retries

`OpenAIChatProvider` works with OpenAI and other OpenAI-compatible hosts. It asks each question with one letter per label. With the
default `logprobs` strategy it reads a distribution from the token probabilities. A reply with no logprobs has no distribution, so
`require_probabilities=True` declines it before any row evidence is sent.

In [22]:
session = MockSession([logprob_reply("A", {"A": 0.8, "B": 0.15})])
chat = OpenAIChatProvider("not-a-real-key", "mock-model", prices=prices, session=session, strategy="logprobs")
with_logprobs = g.sem_decide(suspicious_only, chat, task_id="triage", evidence_cols=["text"], max_rows=2, require_probabilities=True)
print(with_logprobs._nodes[["id", "suspicious", "suspicious__probability_kind", "suspicious__status"]].head(2))
print("dist:", with_logprobs._nodes["suspicious__dist"].iloc[0])

   id  suspicious suspicious__probability_kind suspicious__status
0  a1        True                token_logprob                 ok
1  a2        True                token_logprob                 ok
dist: {"true":0.8421052631578947,"false":0.15789473684210525}


In [23]:
session = MockSession([MockResponse(200, {"model": "mock-chat", "choices": [
    {"message": {"role": "assistant", "content": '{"answer": "A"}'}, "finish_reason": "stop"}]})])
json_chat = OpenAIChatProvider("not-a-real-key", "mock-model", session=session, strategy="json_schema")
value_only = g.sem_decide(suspicious_only, json_chat, task_id="triage", evidence_cols=["text"], max_rows=2)
sent_before = len(session.posts)
declined = g.sem_decide(suspicious_only, json_chat, task_id="triage", evidence_cols=["text"], max_rows=2, require_probabilities=True)
print("value-only answers      :", value_only._nodes["suspicious"].head(2).tolist(), "| requests:", sent_before)
print("with the requirement    :", declined._nodes["suspicious__error_code"].head(2).tolist(),
      "| requests sent by this call:", len(session.posts) - sent_before)

value-only answers      : [True, True] | requests: 2
with the requirement    : ['probabilities_unavailable', 'probabilities_unavailable'] | requests sent by this call: 0


### Reply retries

A reply with no readable answer, or one that is cut off at `max_answer_tokens`, is sent again, up to `reply_retries` more times. Every
request is billed. `cost_usd` and `latency_ms` are the sums, and `retry_count` counts the extra requests. Retries are not part of the cost
estimate or of `budget.max_requests`. A reply that stays unread is an `invalid_provider_result` or `answer_truncated` error that still
reports the billed cost.

In [24]:
one = alerts.head(1)
for retries in (2, 0):
    session = MockSession([text_reply("I am not sure"), text_reply("B")])
    chat = OpenAIChatProvider("not-a-real-key", "mock-model", prices=prices, session=session, strategy="text",
                              reply_retries=retries, retry_backoff_s=0)
    out = graphistry.nodes(one, "id").sem_decide(suspicious_only, chat, task_id="triage", evidence_cols=["text"])
    meta = json.loads(out._nodes["suspicious__meta"].iloc[0])
    print(f"reply_retries={retries}: answer={out._nodes['suspicious'].iloc[0]!s:5} "
          f"error={out._nodes['suspicious__error_code'].iloc[0]!s:24} requests={len(session.posts)} "
          f"retry_count={meta['retry_count']} cost=${meta['cost_usd']:.6f}")

reply_retries=2: answer=False error=None                     requests=2 retry_count=1 cost=$0.000400
reply_retries=0: answer=<NA>  error=invalid_provider_result  requests=1 retry_count=0 cost=$0.000200


## Anthropic Messages

`AnthropicMessagesProvider` reads a code from the reply text. It does not return token probabilities, so its answers have no
distribution. A reply may reason first. The provider reads the answer when the last line, after a blank line, is the code.

In [25]:
session = MockSession([anthropic_reply("The alert shows many failed logins.\n\nA")])
claude = AnthropicMessagesProvider("not-a-real-key", "mock-model", session=session)
out = graphistry.nodes(one, "id").sem_decide(suspicious_only, claude, task_id="triage", evidence_cols=["text"])
out._nodes[["id", "suspicious", "suspicious__probability_kind", "suspicious__status"]]

,id,suspicious,suspicious__probability_kind,suspicious__status
0,a1,True,none,ok


## Live providers (off by default)

Each template below reads its key from an environment variable. Do not put a key in a notebook. The cells run only when you set
`GFQL_NOTEBOOK_LIVE=1`. A live run sends the alert text to the provider and is billed by the provider. Set a `budget`, plan the run with
`explain_sem_decide`, and check your data policy first.

In [26]:
import os

LIVE = os.environ.get("GFQL_NOTEBOOK_LIVE") == "1"
print("live provider cells are", "ON" if LIVE else "off (set GFQL_NOTEBOOK_LIVE=1 to turn them on)")

live provider cells are off (set GFQL_NOTEBOOK_LIVE=1 to turn them on)


In [27]:
# OpenAI Decisions API: native probabilities. Set OPENAI_API_KEY and OPENAI_DECISIONS_MODEL.
if LIVE and os.environ.get("OPENAI_API_KEY"):
    live = OpenAIDecisionsProvider(
        os.environ["OPENAI_API_KEY"], os.environ["OPENAI_DECISIONS_MODEL"],
        prices=TokenPrices(input=float(os.environ["PRICE_IN"]), cached_input=float(os.environ["PRICE_CACHED"])),
        rows_per_request=8)
    result = g.sem_decide(QUESTIONS, live, task_id="triage", evidence_cols=["text"],
                          budget={"max_cost_usd": 0.50, "max_requests": 50}, deadline_s=120)
    display(result._nodes[["id", "suspicious", "suspicious__p", "suspicious__status"]])

In [28]:
# OpenAI-compatible chat (OpenAI, Azure OpenAI, Fireworks, Mistral, vLLM): token-logprob probabilities.
# strategy="auto" probes the host once per endpoint, model and settings; probe requests count against the budget.
if LIVE and os.environ.get("OPENAI_API_KEY"):
    live = OpenAIChatProvider(
        os.environ["OPENAI_API_KEY"], os.environ["OPENAI_CHAT_MODEL"],
        base_url=os.environ.get("OPENAI_BASE_URL", "https://api.openai.com/v1"),
        strategy="auto", prices=TokenPrices(input=float(os.environ["PRICE_IN"]), cached_input=float(os.environ["PRICE_CACHED"])))
    result = g.sem_decide(QUESTIONS, live, task_id="triage", evidence_cols=["text"], budget={"max_requests": 60},
                          require_probabilities=True)
    display(result._nodes[["id", "suspicious", "suspicious__probability_kind", "suspicious__status"]])

In [29]:
# Anthropic Messages: value-only answers. Set ANTHROPIC_API_KEY and ANTHROPIC_MODEL.
if LIVE and os.environ.get("ANTHROPIC_API_KEY"):
    live = AnthropicMessagesProvider(os.environ["ANTHROPIC_API_KEY"], os.environ["ANTHROPIC_MODEL"], strategy="json_schema")
    result = g.sem_decide(QUESTIONS, live, task_id="triage", evidence_cols=["text"], budget={"max_requests": 60})
    display(result._nodes[["id", "suspicious", "suspicious__status"]])

In [30]:
# LangChain: wrap a chat model that you install and configure yourself (for example langchain-openai).
if LIVE and os.environ.get("OPENAI_API_KEY"):
    from langchain_openai import ChatOpenAI
    from graphistry.compute.semantic import LangChainDecisionProvider

    live = LangChainDecisionProvider(ChatOpenAI(model=os.environ["OPENAI_CHAT_MODEL"]), model="deployment",
                                     strategy="json_schema")
    result = g.sem_decide(QUESTIONS, live, task_id="triage", evidence_cols=["text"], budget={"max_requests": 60})
    display(result._nodes[["id", "suspicious", "suspicious__status"]])

## Defaults to check in review

| setting | default | where |
|---|---|---|
| `max_answer_tokens` | 256 | chat providers; output tokens are not in the cost estimate, so lower it to bound them |
| `reply_retries` | 2 | OpenAI-compatible, decisions-endpoint and Anthropic providers; `0` sends once |
| `batch_size` | the provider's `max_rows_per_request`, else 32 (16 for `sem_map`) | `sem_decide`, `sem_map` |
| `rows_per_request` | 1 | `OpenAIDecisionsProvider`; packing is opt in |
| `dedupe` | `True` | `sem_decide`, `sem_map`; `False` asks for every row |
| result cache | none | bind one with `bind_providers(..., cache=...)`; `MemorySemanticCache` keeps 10,000 entries for 1 hour |
| probe requests | count against `budget.max_requests` and `budget.max_cost_usd` | `OpenAIChatProvider(strategy="auto")` |
| `require_probabilities` | `False` | `True` needs measured probabilities for every answer |

A cost cap uses an *estimate* of the input cost. It does not bound output, retry or tracing charges.